<a href="https://colab.research.google.com/github/jadenfix/machine_learning/blob/main/Ensemble_Methods_for_Breast_Cancer_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    BaggingClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    StackingClassifier,
    RandomForestClassifier
)
from sklearn.metrics import accuracy_score, classification_report
import time

# --- 1. Load Built-in Dataset ---
# Load the Breast Cancer Wisconsin (Diagnostic) dataset from Scikit-learn
cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = pd.Series(cancer.target, name='target') # 0: malignant, 1: benign

print("--- Dataset Information ---")
print(f"Loaded {X.shape[0]} samples with {X.shape[1]} features.")
print(f"Feature names: {list(X.columns)}")
print(f"Target names: {list(cancer.target_names)}") # 0: malignant, 1: benign
print(f"Target distribution:\n{y.value_counts(normalize=True)}")
print("-------------------------\n")


# --- 2. Data Preparation ---
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y # Stratify is good practice
)

# Scale features (important for many algorithms, especially Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- 3. Model Definitions ---

# Define base estimators for stacking
# Using simpler models as base learners is common
base_estimators = [
    ('nb', GaussianNB()),
    ('dt', DecisionTreeClassifier(max_depth=5, random_state=42)),
    # Increased max_iter for convergence
    ('lr', LogisticRegression(solver='liblinear', max_iter=1000, random_state=42))
]

# Define meta-learner for stacking
# Increased max_iter for convergence
meta_learner = LogisticRegression(solver='liblinear', max_iter=1000, random_state=42)

# --- 4. Ensemble Model Training and Evaluation ---

models = {}
results = {}

# Function to train and evaluate a model
def train_evaluate(name, model, X_train_data, y_train_data, X_test_data, y_test_data):
    """Trains, evaluates, and times a given Scikit-learn model."""
    print(f"--- Training {name} ---")
    start_time = time.time()
    model.fit(X_train_data, y_train_data)
    end_time = time.time()
    training_time = end_time - start_time
    print(f"Training time: {training_time:.4f} seconds")

    start_time = time.time()
    y_pred = model.predict(X_test_data)
    end_time = time.time()
    prediction_time = end_time - start_time
    print(f"Prediction time: {prediction_time:.4f} seconds")

    accuracy = accuracy_score(y_test_data, y_pred)
    # Use target names for report clarity
    report = classification_report(y_test_data, y_pred, target_names=cancer.target_names, zero_division=0)

    print(f"Accuracy: {accuracy:.4f}")
    print("Classification Report:")
    print(report)
    print("-"*(len("Classification Report:") + len(name) + 2)) # Dynamic divider

    models[name] = model
    results[name] = {'accuracy': accuracy, 'report': report, 'train_time': training_time, 'predict_time': prediction_time}

# --- Bagging ---
bagging_clf = BaggingClassifier(
    # Using Logistic Regression as base estimator this time for variety
    estimator=LogisticRegression(solver='liblinear', max_iter=500, random_state=42),
    n_estimators=100,
    max_samples=0.8,
    max_features=0.8,
    random_state=42,
    n_jobs=-1
)
# Note: Bagging with LR might not always outperform single LR significantly
# if the base model is already stable. Often used with high-variance models like Trees.
train_evaluate("Bagging (Logistic Regression)", bagging_clf, X_train_scaled, y_train, X_test_scaled, y_test)

# --- Boosting ---
# AdaBoost
adaboost_clf = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1), # Default base estimator is Decision Tree Stump
    n_estimators=100,
    learning_rate=0.5, # Adjusted learning rate
    random_state=42
)
train_evaluate("AdaBoost (Decision Tree Stump)", adaboost_clf, X_train_scaled, y_train, X_test_scaled, y_test)

# Gradient Boosting
gradientboost_clf = GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.1, # Slightly higher learning rate than before
    max_depth=3,
    subsample=0.8, # Stochastic Gradient Boosting
    random_state=42
)
train_evaluate("Gradient Boosting", gradientboost_clf, X_train_scaled, y_train, X_test_scaled, y_test)

# --- Stacking ---
stacking_clf = StackingClassifier(
    estimators=base_estimators,
    final_estimator=meta_learner,
    cv=5,
    n_jobs=-1
)
train_evaluate("Stacking (NB, DT, LR -> LR)", stacking_clf, X_train_scaled, y_train, X_test_scaled, y_test)


# --- Random Forest ---
rf_clf = RandomForestClassifier(
    n_estimators=150,
    max_depth=10,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced' # Good practice if classes might be imbalanced
)
train_evaluate("Random Forest", rf_clf, X_train_scaled, y_train, X_test_scaled, y_test)

# --- 5. Summary ---
print("\n--- Model Performance Summary ---")
print(f"{'Model':<35} | {'Accuracy':<10} | {'Train Time (s)':<15} | {'Predict Time (s)':<15}")
print("-" * 80)
for name, result in results.items():
    print(f"{name:<35} | {result['accuracy']:.4f}{'':<4} | {result['train_time']:.4f}{'':<11} | {result['predict_time']:.4f}")
print("---------------------------------")

--- Dataset Information ---
Loaded 569 samples with 30 features.
Feature names: ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness', 'mean compactness', 'mean concavity', 'mean concave points', 'mean symmetry', 'mean fractal dimension', 'radius error', 'texture error', 'perimeter error', 'area error', 'smoothness error', 'compactness error', 'concavity error', 'concave points error', 'symmetry error', 'fractal dimension error', 'worst radius', 'worst texture', 'worst perimeter', 'worst area', 'worst smoothness', 'worst compactness', 'worst concavity', 'worst concave points', 'worst symmetry', 'worst fractal dimension']
Target names: [np.str_('malignant'), np.str_('benign')]
Target distribution:
target
1    0.627417
0    0.372583
Name: proportion, dtype: float64
-------------------------

--- Training Bagging (Logistic Regression) ---
Training time: 13.3585 seconds
Prediction time: 0.3695 seconds
Accuracy: 0.9825
Classification Report:
              precisio